# CS2309 — SwiftEdit WebUI (Gradio)

Setup + chạy **demo web** [`scripts/app_gradio.py`](../scripts/app_gradio.py) trên **Mac MPS** hoặc **Google Colab T4**.

### Hai tab ứng dụng

| Tab | Chức năng |
|-----|----------|
| **Chỉnh sửa bằng prompt** | Upload ảnh + source/edit prompt (semantic edit) |
| **Xóa vật thể (khoanh vùng)** | Cọ tô vùng cần xóa + prompt mô tả nền |

Tích hợp **fp16 + channels_last + EditCache** (SwiftEdit-RT).

### Mac (local)

1. Kernel **`.venv`** (Python 3.12), repo đã clone local
2. Cell **1** nhận `PROJECT_ROOT` → Cell **2** setup → Cell **3** mở WebUI tại `http://127.0.0.1:7860`

### Google Colab (T4)

1. **GPU T4** — Colab web: Runtime → T4; **extension:** New Colab Server → GPU → T4
2. Cell **1** clone repo → `/content/CS2309.CH201` (không chỉ upload `.ipynb` lẻ)
3. Cell **2** pip + weights (~10GB lần đầu)
4. Cell **3** launch Gradio với **`share=True`** → link `*.gradio.live` mở trên máy bạn

Repo private: Colab Secrets → `GITHUB_TOKEN`. Chỉnh `REPO_SLUG` ở cell 1 nếu fork.

### ① Clone / nhận repo + kiểm tra GPU (Colab)

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


try:
    import ipywidgets as widgets
except ImportError:
    !pip install ipywidgets
    import ipywidgets as widgets
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


_COLAB_GPU_ERR = (
    "Colab chưa có GPU.\n"
    "• Colab web: Runtime → Change runtime type → T4 GPU\n"
    "• Colab extension: Select Kernel → Colab → New Colab Server "
    "→ Hardware accelerator: GPU → T4, rồi Restart kernel\n"
    "• Đang nối server CPU: Remove Server, tạo server GPU mới"
)


def _check_colab_gpu() -> None:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0 or not (r.stdout or "").strip():
        raise RuntimeError(_COLAB_GPU_ERR)
    names = [ln.strip() for ln in r.stdout.strip().splitlines() if ln.strip()]
    print("GPU OK (nvidia-smi):", ", ".join(names))


REPO_SLUG = "NguyenKz/CS2309.CH201"
USE_PRIVATE_REPO = True


def _colab_repo_url():
    public_url = f"https://github.com/{REPO_SLUG}.git"
    if not USE_PRIVATE_REPO:
        return public_url
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        if not token:
            print("Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.")
            raise ValueError("Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.")
        return f"https://{token}@github.com/{REPO_SLUG}.git"
    except Exception as e:
        text_input = widgets.Text(value='Enter GITHUB_TOKEN', description='Text:')
        token = text_input.value
        if token:
            return f"https://{token}@github.com/{REPO_SLUG}.git"
        else:
            print(
                "Không lấy GITHUB_TOKEN — dùng repo public hoặc thêm secret Colab.\n"
                f"Chi tiết: {e}\nFallback: {public_url}"
            )
    raise e

REPO_URL = _colab_repo_url() if IN_COLAB else f"https://github.com/{REPO_SLUG}.git"
COLAB_REPO_DIR = Path("/content/CS2309.CH201")

if IN_COLAB:
    _check_colab_gpu()
    if not (COLAB_REPO_DIR / "SwiftEdit" / "infer.py").exists():
        print(f"Cloning github.com/{REPO_SLUG} → {COLAB_REPO_DIR} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(COLAB_REPO_DIR)],
            check=True,
        )
    PROJECT_ROOT = COLAB_REPO_DIR
    os.environ.setdefault("HF_HOME", "/content/huggingface")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")
else:
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent
    elif not (PROJECT_ROOT / "SwiftEdit" / "infer.py").exists():
        for p in [Path.cwd(), *Path.cwd().parents]:
            if (p / "SwiftEdit" / "infer.py").exists():
                PROJECT_ROOT = p
                break

APP_SCRIPT = PROJECT_ROOT / "scripts" / "app_gradio.py"
print("IN_COLAB:", IN_COLAB)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("app_gradio.py:", APP_SCRIPT.is_file())
print("Weights OK:", (PROJECT_ROOT / "SwiftEdit/swiftedit_weights/inverse_ckpt-120k").is_dir())
if IN_COLAB and not (PROJECT_ROOT / "SwiftEdit" / "infer.py").is_file():
    raise FileNotFoundError("Clone xong nhưng thiếu SwiftEdit/ — push đủ repo lên GitHub.")

### ② Setup — pip, weights, HF, Gradio

In [ ]:
env = os.environ.copy()
env["REPO_SLUG"] = REPO_SLUG
env["COLAB_REPO_DIR"] = str(COLAB_REPO_DIR)
if IN_COLAB:
    env.setdefault("HF_HOME", "/content/huggingface")
    env.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "900")

if IN_COLAB:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_colab.sh"
else:
    setup_sh = PROJECT_ROOT / "scripts" / "setup_macos.sh"

print("Chạy:", setup_sh)
subprocess.run(["bash", str(setup_sh)], check=True, env=env, cwd=PROJECT_ROOT)

# Gradio + nest_asyncio (WebUI trong notebook)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gradio>=5,<6", "huggingface-hub<1.0", "nest-asyncio"],
    check=True,
)
import gradio as gr

print("gradio:", gr.__version__)
print("Setup OK — chạy cell Launch WebUI.")

### ③ Launch WebUI

- **Local:** mở `http://127.0.0.1:7860` (hoặc đổi `PORT`)
- **Colab:** dùng link **`*.gradio.live`** (tự bật `share=True`)
- Dừng server: **Interrupt kernel** (■) hoặc Runtime → Restart

Đổi `DTYPE`: `"fp16"` (mặc định, nhanh) hoặc `"fp32"` (so sánh chất lượng).

In [ ]:
import nest_asyncio

nest_asyncio.apply()

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from app_gradio import build_app

# --- Cấu hình ---
DTYPE = "fp16"  # "fp32" để so baseline
PORT = 7860
SHARE = IN_COLAB  # Colab: link public gradio.live
SERVER_NAME = "0.0.0.0" if IN_COLAB else "127.0.0.1"

print(f"Đang nạp model ({DTYPE}) — lần đầu có thể mất vài phút...")
demo, _, _ = build_app(DTYPE)

print("\n" + "=" * 60)
if IN_COLAB:
    print("Colab: mở link *.gradio.live bên dưới (share=True).")
else:
    print(f"Local: mở http://127.0.0.1:{PORT}")
print("Tab 1: Chỉnh sửa bằng prompt  |  Tab 2: Xóa vật thể (khoanh vùng)")
print("=" * 60 + "\n")

demo.launch(
    share=SHARE,
    server_name=SERVER_NAME,
    server_port=PORT,
    show_error=True,
    prevent_thread_lock=True,
)